# TensorFlow Scala Example


First we have to load [TensorFlowScala](http://platanios.org/tensorflow_scala) into our kernel.

In [1]:
interp.load.ivy(
  // replace with darwin on macOS
  ("org.platanios" %% "tensorflow" % "0.6.5").withClassifier("linux"),
  "org.platanios" %% "tensorflow-data" % "0.6.5"
)

In [2]:
import org.platanios.tensorflow.api._
import org.platanios.tensorflow.api.learn._
import org.platanios.tensorflow.api.learn.layers._
import org.platanios.tensorflow.api.learn.estimators.InMemoryEstimator
import org.platanios.tensorflow.data.image.MNISTLoader

import java.nio.file.Paths
import scala.util.Random

import org.platanios.tensorflow.api._
import org.platanios.tensorflow.api.learn._
import org.platanios.tensorflow.api.learn.layers._
import org.platanios.tensorflow.api.learn.estimators.InMemoryEstimator
import org.platanios.tensorflow.data.image.MNISTLoader
import java.nio.file.Paths
import scala.util.Random

The original location of the MNIST dataset (on yann.lecun.com) isn't available anymore, so we download it from a mirror first. `MNISTLoader` then finds the files, and doesn't try to download them on its own.

In [3]:
val mnistDir = Paths.get("tmp/mnist")

locally {
  java.nio.file.Files.createDirectories(mnistDir)
  for (name <- Seq("train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz", "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz")) {
    val dest = mnistDir.resolve(name)
    if (!java.nio.file.Files.exists(dest)) {
      val in = java.net.URI.create(s"https://storage.googleapis.com/cvdf-datasets/mnist/$name").toURL.openStream()
      try java.nio.file.Files.copy(in, dest)
      finally in.close()
    }
  }
}

mnistDir: java.nio.file.Path = tmp/mnist

In [4]:
val dataset = MNISTLoader.load(mnistDir)

2026-10-22 14:50:34.129 [scala-kernel-interpreter-1] INFO  MNIST Data Loader - Extracting images from file 'tmp/mnist/train-images-idx3-ubyte.gz'.
2026-10-22 14:50:44.091 [scala-kernel-interpreter-1] INFO  MNIST Data Loader - Extracting labels from file 'tmp/mnist/train-labels-idx1-ubyte.gz'.
2026-10-22 14:50:44.093 [scala-kernel-interpreter-1] INFO  MNIST Data Loader - Extracting images from file 'tmp/mnist/t10k-images-idx3-ubyte.gz'.
2026-10-22 14:50:44.139 [scala-kernel-interpreter-1] INFO  MNIST Data Loader - Extracting labels from file 'tmp/mnist/t10k-labels-idx1-ubyte.gz'.
2026-10-22 14:50:44.140 [scala-kernel-interpreter-1] INFO  MNIST Data Loader - Finished loading the MNIST dataset.


dataset: org.platanios.tensorflow.data.image.MNISTDataset = MNISTDataset(
  datasetType = MNIST,
  trainImages = Tensor[UByte, [60000, 28, 28]],
  trainLabels = Tensor[UByte, [60000]],
  testImages = Tensor[UByte, [10000, 28, 28]],
  testLabels = Tensor[UByte, [10000]]
)

Let's display a few example images from our training set.

In [5]:
val sess = Session()
def showImage(image: Tensor[UByte]): Unit = {
  val exampleImage = tf.decodeRaw[Byte](tf.image.encodePng(image))
  val png = sess.run(fetches = exampleImage)
  Image(png.entriesIterator.toArray).withFormat(Image.PNG).withWidth(50).display()
}

val randomImages = for {
  index <- (0 to 4).map(_ => Random.nextInt(dataset.testImages.shape(0)))
  image = dataset.testImages(index).expandDims(-1)
} yield image
randomImages.foreach(showImage(_))

sess: Session = org.platanios.tensorflow.api.core.client.Session@20b080bc
defined function showImage
randomImages: IndexedSeq[Tensor[UByte]] = Vector(
  Tensor[UByte, [28, 28, 1]],
  Tensor[UByte, [28, 28, 1]],
  Tensor[UByte, [28, 28, 1]],
  Tensor[UByte, [28, 28, 1]],
  Tensor[UByte, [28, 28, 1]]
)

In [6]:
// Load and batch data using pre-fetching.
val trainImages = tf.data.datasetFromTensorSlices(dataset.trainImages.toFloat)
val trainLabels = tf.data.datasetFromTensorSlices(dataset.trainLabels.toLong)
val trainData =
  trainImages.zip(trainLabels)
      .repeat()
      .shuffle(10000)
      .batch(256)
      .prefetch(10)

trainImages: ops.data.Dataset[Output[Float]] = Dataset[TensorSlicesDataset]
trainLabels: ops.data.Dataset[Output[Long]] = Dataset[TensorSlicesDataset]
trainData: ops.data.Dataset[(Output[Float], Output[Long])] = Dataset[TensorSlicesDataset/Zip/Repeat/Shuffle/Batch/Prefetch]

In [7]:
// Create the MLP model.
val input = Input(FLOAT32, Shape(-1, 28, 28))
val trainInput = Input(INT64, Shape(-1))
val layer = Flatten[Float]("Input/Flatten") >>
    Linear[Float]("Layer_0", 128) >> ReLU[Float]("Layer_0/Activation", 0.1f) >>
    Linear[Float]("Layer_1", 64) >> ReLU[Float]("Layer_1/Activation", 0.1f) >>
    Linear[Float]("Layer_2", 32) >> ReLU[Float]("Layer_2/Activation", 0.1f) >>
    Linear[Float]("OutputLayer", 10)
val loss = SparseSoftmaxCrossEntropy[Float, Long, Float]("Loss") >>
    Mean("Loss/Mean")
val optimizer = tf.train.GradientDescent(1e-6f)
val model = Model.simpleSupervised(input, trainInput, layer, loss, optimizer)

input: Input[Output[Float]] = org.platanios.tensorflow.api.learn.layers.Input@60399b99
trainInput: Input[Output[Long]] = org.platanios.tensorflow.api.learn.layers.Input@1b684aeb
layer: Compose[Output[Float], Output[Float], Output[Float]] = Compose(
  name = "Input/Flatten",
  layer1 = Compose(
    name = "Input/Flatten",
    layer1 = Compose(
      name = "Input/Flatten",
      layer1 = Compose(
        name = "Input/Flatten",
        layer1 = Compose(
          name = "Input/Flatten",
          layer1 = Compose(
            name = "Input/Flatten",
            layer1 = Compose(
              name = "Input/Flatten",
              layer1 = Flatten(name = "Input/Flatten"),
              layer2 = Linear(
                name = "Layer_0",
                units = 128,
                useBias = true,
                weightsInitializer = RandomNormalInitializer(
                  mean = Tensor[Float, []],
                  standardDeviation = Tensor[Float, []],
                  seed = None
  

In [8]:
import org.platanios.tensorflow.api.learn.hooks._
import org.platanios.tensorflow.api.config.TensorBoardConfig

val loss = SparseSoftmaxCrossEntropy[Float, Long, Float]("Loss") >>
    Mean("Loss/Mean") >>
    ScalarSummary(name = "Loss", tag = "Loss")
val summariesDir = Paths.get("tmp/summaries")
val estimator = InMemoryEstimator(
  modelFunction = model,
  configurationBase = Configuration(Some(summariesDir)),
  trainHooks = Set(
    SummarySaver(summariesDir, StepHookTrigger(100)),
    CheckpointSaver(summariesDir, StepHookTrigger(1000))),
  tensorBoardConfig = TensorBoardConfig(summariesDir))

2026-10-22 14:50:47.207 [scala-kernel-interpreter-1] INFO  Learn / Hooks / TensorBoard - Launching TensorBoard in 'localhost:6006' for log directory '/home/jovyan/tmp/summaries'.
2026-10-22 14:50:47.211 [scala-kernel-interpreter-1] WARN  Learn / Hooks / TensorBoard - Cannot run program "tensorboard": error=2, No such file or directory
2026-10-22 14:50:47.211 [scala-kernel-interpreter-1] WARN  Learn / Hooks / TensorBoard - Could not launch TensorBoard. Please make sure it is installed correctly and in your PATH.


import org.platanios.tensorflow.api.learn.hooks._
import org.platanios.tensorflow.api.config.TensorBoardConfig
loss: Compose[(Output[Float], Output[Long]), Output[Float], Output[Float]] = Compose(
  name = "Loss",
  layer1 = Compose(
    name = "Loss",
    layer1 = SparseSoftmaxCrossEntropy(name = "Loss"),
    layer2 = Mean(name = "Loss/Mean", axes = null, keepDims = false)
  ),
  layer2 = ScalarSummary(
    name = "Loss",
    tag = "Loss",
    family = null,
    collections = Set(
      org.platanios.tensorflow.api.core.Graph$Keys$SUMMARIES$@9778635
    )
  )
)
summariesDir: java.nio.file.Path = tmp/summaries
estimator: InMemoryEstimator[Output[Float], (Output[Float], Output[Long]), Output[Float], Output[Float], Float, (Output[Float], (Output[Float], Output[Long]))] = org.platanios.tensorflow.api.learn.estimators.InMemoryEstimator@e621419

In [9]:
estimator.train(() => trainData, StopCriteria(maxSteps = Some(10000)))

2026-10-22 14:50:51.283 [scala-kernel-interpreter-1] INFO  Learn / Hooks / Checkpoint Saver - Saving checkpoint for step 0.
2026-10-22 14:50:51.284 [scala-kernel-interpreter-1] INFO  Variables / Saver - Saving parameters to '/home/jovyan/tmp/summaries/model.ckpt-0'.
2026-10-22 14:50:52.788 [scala-kernel-interpreter-1] INFO  Variables / Saver - Saved parameters to '/home/jovyan/tmp/summaries/model.ckpt-0'.
2026-10-22 14:50:54.851 [scala-kernel-interpreter-1] INFO  Learn / Hooks / Checkpoint Saver - Saving checkpoint for step 1000.
2026-10-22 14:50:54.852 [scala-kernel-interpreter-1] INFO  Variables / Saver - Saving parameters to '/home/jovyan/tmp/summaries/model.ckpt-1000'.
2026-10-22 14:50:55.757 [scala-kernel-interpreter-1] INFO  Variables / Saver - Saved parameters to '/home/jovyan/tmp/summaries/model.ckpt-1000'.
2026-10-22 14:50:57.135 [scala-kernel-interpreter-1] INFO  Learn / Hooks / Checkpoint Saver - Saving checkpoint for step 2000.
2026-10-22 14:50:57.135 [scala-kernel-interpre

In [10]:
val images = dataset.testImages(0::10)
val result = estimator.infer(() => images.toFloat)
val predictedClasses = result.argmax(1)
for (i <- 0 until images.shape(0)) {
    showImage(images(i).expandDims(-1))
    print("predicted class: ")
    print(predictedClasses(i).scalar)
}

predicted class: 7

predicted class: 2

predicted class: 1

predicted class: 0

predicted class: 4

...

predicted class: 4

predicted class: 4

predicted class: 6

predicted class: 9

images: Tensor[UByte] = Tensor[UByte, [10, 28, 28]]
result: Tensor[Float] = Tensor[Float, [10, 10]]
predictedClasses: Tensor[Long] = Tensor[Long, [10]]